# Тематический анализ разных текстов отчетов

Конфигурация: указать нужные пути к папкам

In [2]:
# путь к папке с выборкой (JSON файлами) на Google Диске
JSON_FOLDER_PATH = "/content/drive/MyDrive/ВЫБОРКА"

# путь для сохранения результатов извлечения ключевых слов
OUTPUT_KEYWORDS_PATH = "/content/drive/MyDrive/анализ отчетов/extracted_keywords.csv"

# путь к файлу с результатами косинусной близости
COSINE_SIMILARITY_PATH = "/content/drive/MyDrive/анализ отчетов/косинусная близость отчетов.csv"

# путь к файлу с извлеченными ключевыми словами
KEYWORDS_FILE_PATH = "/content/drive/MyDrive/анализ отчетов/extracted_keywords.csv"

In [5]:
!pip install keybert -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.4/41.4 kB 976.9 kB/s eta 0:00:00


In [6]:
import os
import json
import pandas as pd
from keybert import KeyBERT
from google.colab import drive
from collections import Counter

извлечение ключевых слов

In [ ]:
drive.mount('/content/drive')
folder_path = JSON_FOLDER_PATH

data = []

for file in os.listdir(folder_path):
    if file.endswith(".json"):
        with open(os.path.join(folder_path, file), "r", encoding="utf-8") as f:
            try:
                j = json.load(f)
                data.append({
                    "company": j.get("company"),
                    "year": j.get("fiscal_year"),
                    "section_1_text": j.get("section_item_1", ""),
                    "section_1A_text": j.get("section_item_1a", ""),
                    "section_7_text": j.get("section_item_7", "")
                })
            except:
                print("Error in file:", file)

df = pd.DataFrame(data)

def clean_text(text):
    if pd.isna(text):
        return ""
    return str(text).replace("\n", " ").lower()

df['section_1_text'] = df['section_1_text'].apply(clean_text)
df['section_1A_text'] = df['section_1A_text'].apply(clean_text)
df['section_7_text'] = df['section_7_text'].apply(clean_text)

kw_model = KeyBERT()

def extract_keywords(text):
    if not text or len(text.strip()) < 50:
        return []
    try:
        keywords = kw_model.extract_keywords(
            text,
            keyphrase_ngram_range=(1, 2),
            stop_words='english',
            top_n=20
        )
        return [{"keyword": kw[0], "score": float(kw[1])} for kw in keywords]
    except:
        return []

df['keywords_1'] = df['section_1_text'].apply(extract_keywords)
df['keywords_1A'] = df['section_1A_text'].apply(extract_keywords)
df['keywords_7'] = df['section_7_text'].apply(extract_keywords)

df_to_save = df[['company', 'year', 'keywords_1', 'keywords_1A', 'keywords_7']].copy()

df_to_save['keywords_1'] = df_to_save['keywords_1'].apply(lambda x: json.dumps(x, ensure_ascii=False))
df_to_save['keywords_1A'] = df_to_save['keywords_1A'].apply(lambda x: json.dumps(x, ensure_ascii=False))
df_to_save['keywords_7'] = df_to_save['keywords_7'].apply(lambda x: json.dumps(x, ensure_ascii=False))

output_path = OUTPUT_KEYWORDS_PATH
df_to_save.to_csv(output_path, index=False, encoding='utf-8')


Mounted at /content/drive


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Статистика по извлеченным ключевым словам

In [9]:
drive.mount('/content/drive')

keywords_path = KEYWORDS_FILE_PATH

df = pd.read_csv(keywords_path, sep=';', encoding='utf-8')

expected_cols = ['company', 'year', 'keywords_1', 'keywords_1A', 'keywords_7']
if len(df.columns) == len(expected_cols):
    df.columns = expected_cols

df['company'] = df['company'].fillna('berkshire')
df['year'] = pd.to_numeric(df['year'], errors='coerce').fillna(0).astype(int)

def parse_keywords(json_str):
    if pd.isna(json_str) or json_str == '[]' or json_str == '':
        return []
    try:
        fixed_str = json_str.replace('} {', '}, {')
        return json.loads(fixed_str)
    except json.JSONDecodeError as e:
        try:
            return json.loads(json_str)
        except:
            return []

df['keywords_1'] = df['keywords_1'].apply(parse_keywords)
df['keywords_1A'] = df['keywords_1A'].apply(parse_keywords)
df['keywords_7'] = df['keywords_7'].apply(parse_keywords)

def extract_keyword_list(keywords_with_scores):
    if not keywords_with_scores:
        return []
    result = []
    for kw in keywords_with_scores:
        if isinstance(kw, dict) and 'keyword' in kw:
            result.append(kw['keyword'])
        elif isinstance(kw, str):
            result.append(kw)
    return result

df['kw_1'] = df['keywords_1'].apply(extract_keyword_list)
df['kw_1A'] = df['keywords_1A'].apply(extract_keyword_list)
df['kw_7'] = df['keywords_7'].apply(extract_keyword_list)

def topic_overlap(a, b):
    a, b = set(a), set(b)
    if len(a) == 0 or len(b) == 0:
        return 0
    return len(a & b) / len(a | b)

df['topic_overlap_1_7'] = df.apply(lambda x: topic_overlap(x['kw_1'], x['kw_7']), axis=1)
df['topic_overlap_1A_7'] = df.apply(lambda x: topic_overlap(x['kw_1A'], x['kw_7']), axis=1)
df['topic_gap'] = df['topic_overlap_1_7'] - df['topic_overlap_1A_7']


print(" статистика topic overlap ".center(60))
overlap_stats = df[['topic_overlap_1_7', 'topic_overlap_1A_7']].describe().round(4)
print(overlap_stats)

print(" статистика topic gap ".center(60))
gap_stats = df['topic_gap'].describe().round(4)
print(gap_stats)

print(" средние показатели по компаниям ".center(60))
company_stats = df.groupby('company').agg({
    'topic_overlap_1_7': 'mean',
    'topic_overlap_1A_7': 'mean',
    'topic_gap': 'mean',
    'year': 'count'
}).round(4)
company_stats.columns = ['overlap_1_7', 'overlap_1A_7', 'gap', 'num_years']
print(company_stats.sort_values('gap'))

def print_keywords_table(keywords_list, title, n=20):
    counter = Counter()
    for sub in keywords_list:
        if sub:
            counter.update(sub)
    print(title.center(60))
    if counter:
        df_keywords = pd.DataFrame(counter.most_common(n), columns=['Ключевое слово', 'Частота'])
        print(df_keywords.to_string(index=False))
    else:
        print("нет ключевых слов для отображения")
    return counter

counter_1 = print_keywords_table(df['kw_1'], "топ-20 ключевых слов: стратегия (раздел 1)", 20)
counter_1A = print_keywords_table(df['kw_1A'], "топ-20 ключевых слов: риски (раздел 1A)", 20)
counter_7 = print_keywords_table(df['kw_7'], "топ-20 ключевых слов: результаты (раздел 7)", 20)

print(" динамика по годам ".center(60))
yearly_stats = df.groupby('year').agg({
    'topic_overlap_1_7': 'mean',
    'topic_overlap_1A_7': 'mean',
    'topic_gap': 'mean'
}).round(4)
yearly_stats.columns = ['overlap_1_7', 'overlap_1A_7', 'gap']
print(yearly_stats)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
                  статистика topic overlap                  
       topic_overlap_1_7  topic_overlap_1A_7
count            99.0000             99.0000
mean              0.0202              0.0198
std               0.0436              0.0567
min               0.0000              0.0000
25%               0.0000              0.0000
50%               0.0000              0.0000
75%               0.0256              0.0256
max               0.2121              0.4815
                    статистика topic gap                    
count    99.0000
mean      0.0004
std       0.0748
min      -0.4815
25%      -0.0256
50%       0.0000
75%       0.0256
max       0.2121
Name: topic_gap, dtype: float64
              средние показатели по компаниям               
         overlap_1_7  overlap_1A_7     gap  num_years
company                                              
elv    